In [1]:
import os
from getpass import getpass

if "LLAMA_CLOUD_API_KEY" not in os.environ:
    os.environ["LLAMA_CLOUD_API_KEY"] = getpass("Enter your Llama Cloud API Key: ")

## Demo Extraction Schema

In [2]:
from pydantic import BaseModel, Field

class PartSchema(BaseModel):
    spur_gear_material: str = Field(description="The material from which the spur gear was manufactured (e.g. Steel, Stainless Steel, Plastics (Polyketon (PK), Polyacetal (POM)), etc.)")
    straight_toothed: bool = Field(description="It indicates whether, the teeth are aligned longitudinally with the shaft, meaning there is no \"helix angle\".")
    angle_of_engagement: int = Field(description="It refers to the angular position, or the arc, during which two gear teeth are in contact and transmitting power. It is often written in Degrees (°).")
    module: float = Field(description="The gear module of a gear represents the ratio of the pitch (distance between teeth) to pi (\\(\\pi \\)), effectively defining how thick a gear tooth is and, consequently, how strong it is.")

In [3]:
PDF_PATH = "/home/daghbeji/rag-factory/mechanical-parts-catalogs/data/gear_m2.pdf"

## Agent-free extraction for quick prototyping

In [ ]:
from llama_cloud_services import LlamaExtract
from llama_cloud import ExtractConfig

extractor = LlamaExtract()

# Data Schema and Config
json_data = {
    "dataSchema": PartSchema,
    "config": {
        "priority": None,
        "extraction_target": "PER_PAGE",
        "extraction_mode": "PREMIUM",
        "parse_model": "anthropic-haiku-4.5",
        "extract_model": "openai-gpt-4-1",
        "multimodal_fast_mode": False,
        "system_prompt": "You are an expert at extracting specifications of spur gears from catalog documents",
        "use_reasoning": False,
        "citation_bbox": False,
        "confidence_scores": False,
        "chunk_mode": "PAGE",
        "high_resolution_mode": False,
        "invalidate_cache": False,
        "num_pages_context": None,
        "page_range": None,
        "cite_sources": True
    }
}

try:
    # Use schema and config from playground
    data_schema = json_data["dataSchema"]
    config = ExtractConfig(**json_data["config"])

    # Extract data directly from document - no agent needed!
    result = extractor.extract(data_schema, config, PDF_PATH)
    print(result.data)
    
except Exception as e:
    print(f"Error: {e}")

/home/daghbeji/rag-factory/sandbox/new-shit/lib/python3.12/site-packages/llama_cloud_services/extract/extract.py:206: ExperimentalWarning: `cite_sources`/`confidence_scores` could greatly increase the size of the response, and slow down the extraction. Results will be available in the `extraction_metadata` field for the extraction run.
  warnings.warn(


[{'spur_gear_material': 'Polyacetal (POM)', 'straight_toothed': True, 'angle_of_engagement': 20, 'module': 2.0}]


## Stateless one-shot extraction (for one/few docs with no reusable setup)

In [8]:
from __future__ import annotations
import asyncio
from typing import Optional

from pydantic import Field, BaseModel
from llama_cloud import AsyncLlamaCloud

class Models(BaseModel):
    model_names: Optional[list[str]] = Field(description="List of models mentioned.")

async def extract_stateless() -> None:
    client = AsyncLlamaCloud()

    file_obj = await client.files.create(
        file=PDF_PATH,
        purpose="extract",
    )
    file_id = file_obj.id

    # Stateless one-shot extraction
    result = await client.extraction.extract(
        file_id=file_id,
        config={
            "chunk_mode": "PAGE",
            "cite_sources": True,
            "extraction_target": "PER_DOC",
            "extraction_mode": "PREMIUM",
        },
        data_schema=Models.model_json_schema(),
    )

    extracted_model = Models.model_validate(result.data)
    print("Extracted model names:", extracted_model.model_names)

stateless_extraction_result = await extract_stateless() 

ImportError: cannot import name 'AsyncLlamaCloud' from 'llama_cloud' (/home/daghbeji/rag-factory/sandbox/new-shit/lib/python3.12/site-packages/llama_cloud/__init__.py)

## Agent-based extraction (reusable extractor for document family):
For example:
- 200 mechanical-parts catalog PDFs from the same manufacturer family<br>
- For all of them, you always want to extract:<br>
    - part number<br>
    - DIN standard<br>
    - module<br>
    - material<br>
    - table-row dimension

In [9]:
async def extract_with_agent() -> None:
    client = AsyncLlamaCloud()

    file_obj = await client.files.create(
        file=PDF_PATH,
        purpose="extract",
    )
    file_id = file_obj.id

    # Create an extraction agent
    agent = await client.extraction.extraction_agents.create(
        config={
            "chunk_mode": "PAGE",
            "cite_sources": True,
            "extraction_target": "PER_DOC",
            "extraction_mode": "BALANCED",
        },
        data_schema=Models.model_json_schema(),
        name="My Extraction Agent",
    )

    # Use the extraction agent
    result = await client.extraction.jobs.extract(
        extraction_agent_id=agent.id,
        file_id=file_id,
    )

    extracted_model = Models.model_validate(result.data)
    print("Extracted model names with agent:", extracted_model.model_names)


agent_extraction_result = await extract_with_agent()

NameError: name 'AsyncLlamaCloud' is not defined